# Differentiated Thyroid Cancer Recurrence: Statistical Analysis and Predictive Modeling
**Research question:** Can demographic and clinical characteristics help predict differentiated thyroid cancer recurrence?

**Dataset:** [UCI Machine Learning Repository, dataset 915](https://archive.ics.uci.edu/dataset/915/differentiated+thyroid+cancer+recurrence)

**Important:** This is an educational retrospective analysis, not a validated clinical decision tool. The dataset may include variables observed after diagnosis. In particular, `Response` is excluded from the primary analysis to avoid using post-treatment information when answering a diagnosis-time prediction question. Clinical staging/risk fields should also be verified against the intended prediction time in any real deployment.

Run cells in order in Google Colab. The first data cell reads the CSV in the repository, or downloads the original archive when online.

## 1. Setup and reproducibility
**Enhancement 1:** Centralized configuration and fixed random seed. This makes experiments easier to reproduce and rerun.

In [ ]:
import io, zipfile, urllib.request, warnings, re
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance
SEED = 42
TEST_SIZE = 0.25
sns.set_theme(style='whitegrid')
np.random.seed(SEED)
print('Setup complete')

## 2. Import and validate data
**Enhancement 2:** Flexible local-file/archive loading with explicit validation, normalized column names, duplicate reporting, and clear failure messages. The repository's `data/` folder is preferred to ensure repeatability.

In [ ]:
DATA_PATH = Path('data/Thyroid_Diff.csv')
ARCHIVE_URL = 'https://archive.ics.uci.edu/static/public/915/differentiated+thyroid+cancer+recurrence.zip'

def load_dataset(path=DATA_PATH):
    if path.exists():
        print(f'Reading local dataset: {path}')
        return pd.read_csv(path)
    print('Local CSV not found; downloading original UCI archive...')
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=30) as response:
            raw = response.read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            csv_names = [name for name in z.namelist() if name.lower().endswith('.csv')]
            if not csv_names:
                raise ValueError('No CSV file found in the UCI archive.')
            with z.open(csv_names[0]) as f:
                data = pd.read_csv(f)
        path.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(path, index=False)
        print(f'Saved {path}')
        return data
    except Exception as exc:
        raise RuntimeError('Dataset unavailable. Download the CSV from the UCI page and place it at data/Thyroid_Diff.csv, or upload the repository data folder to Colab.') from exc

raw_df = load_dataset()
print('Original shape:', raw_df.shape)
print('Original columns:', raw_df.columns.tolist())
raw_df.head()

In [ ]:
def normalize_name(name):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', str(name).strip().lower())).strip('_')

df = raw_df.copy()
df.columns = [normalize_name(c) for c in df.columns]
if len(set(df.columns)) != len(df.columns):
    raise ValueError('Duplicate column names after normalization; inspect the original CSV.')
if 'recurred' not in df.columns:
    raise ValueError(f'Expected target column "Recurred". Found: {df.columns.tolist()}')
if df.empty:
    raise ValueError('Dataset is empty.')
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].astype('string').str.strip().replace({'': pd.NA, '?': pd.NA, 'nan': pd.NA})
print('Exact duplicate rows:', df.duplicated().sum())
print('Missing values by column:')
print(df.isna().sum().sort_values(ascending=False).to_string())
print('Target values:', df['recurred'].value_counts(dropna=False).to_dict())

## 3. Cleaning, leakage review, and feature engineering
**Enhancement 3:** Explicit exclusion of treatment response (potential target leakage), duplicate removal, target validation, and an age-group feature for descriptive analysis. The modeling age variable remains continuous to preserve information.

In [ ]:
df = df.drop_duplicates().copy()
# Exclude records without a valid target rather than silently assuming their label.
target_text = df['recurred'].astype('string').str.strip().str.lower()
target_map = {'yes': 1, 'no': 0, '1': 1, '0': 0, 'true': 1, 'false': 0}
y_all = target_text.map(target_map)
if y_all.isna().any():
    raise ValueError(f'Unknown/missing recurrence labels: {df.loc[y_all.isna(), "recurred"].unique().tolist()}')
df['recurrence_binary'] = y_all.astype(int)
if df['recurrence_binary'].nunique() != 2:
    raise ValueError('Both recurrence classes are required for binary classification.')
if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df.loc[~df['age'].between(0, 110), 'age'] = np.nan
    df['age_group'] = pd.cut(df['age'], bins=[-np.inf, 30, 45, 60, np.inf], labels=['≤30', '31–45', '46–60', '>60'])
leakage_cols = [c for c in df.columns if c in {'response', 'recurred', 'recurrence_binary', 'age_group'}]
X = df.drop(columns=leakage_cols)
y = df['recurrence_binary']
print('Excluded from modeling:', leakage_cols)
print('Modeling features:', X.columns.tolist())
print('Final rows:', len(df))

## 4. Descriptive statistics and visual exploration
**Enhancement 4:** Automated recurrence-rate plots across available clinical variables, including outcome imbalance and stratified age distribution. Missing optional columns are handled gracefully.

In [ ]:
display(df.describe(include='all').T)
counts = y.value_counts().sort_index()
print('Class distribution:', counts.to_dict())
print('Recurrence prevalence:', f'{y.mean():.1%}')
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(data=df, x='recurrence_binary', order=[0,1], ax=ax)
ax.set_xticklabels(['No recurrence','Recurrence'])
ax.set(title='Recurrence outcome distribution', xlabel='Outcome', ylabel='Patients')
plt.tight_layout(); plt.show()
if 'age' in df:
    plt.figure(figsize=(7,4))
    sns.boxplot(data=df, x='recurrence_binary', y='age', order=[0,1])
    plt.xticks([0,1], ['No recurrence','Recurrence']); plt.title('Age by recurrence status')
    plt.tight_layout(); plt.show()

In [ ]:
plot_vars = [c for c in ['risk', 't', 'n', 'm', 'stage', 'pathology', 'gender', 'smoking', 'age_group'] if c in df]
for feature in plot_vars:
    summary = (df.groupby(feature, observed=True, dropna=False)['recurrence_binary']
                 .agg(rate='mean', patients='size').reset_index())
    summary[feature] = summary[feature].astype(str)
    fig, ax = plt.subplots(figsize=(max(6, min(13, len(summary)*1.1)), 4))
    sns.barplot(data=summary, x=feature, y='rate', color='steelblue', ax=ax)
    ax.set(title=f'Recurrence rate by {feature} (descriptive)', ylabel='Observed recurrence rate', xlabel=feature)
    ax.set_ylim(0,1)
    ax.tick_params(axis='x', rotation=40)
    plt.tight_layout(); plt.show()
    display(summary.sort_values('rate', ascending=False))

## 5. Inferential statistical analysis
**Enhancement 5:** Automated Chi-square tests for categorical predictors, Cramér's V effect sizes, and a Mann–Whitney U test for age. Tests are exploratory and do not establish causation. Sparse cells can weaken Chi-square approximations; p-values are unadjusted for multiple testing.

In [ ]:
def cramers_v(table):
    chi2, _, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    denom = min(table.shape[0]-1, table.shape[1]-1)
    return float(np.sqrt((chi2/n)/denom)) if n > 0 and denom > 0 else np.nan

test_results = []
for feature in plot_vars:
    if feature == 'age_group':
        continue
    table = pd.crosstab(df[feature], y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        continue
    chi2, pval, dof, expected = chi2_contingency(table)
    test_results.append({'feature':feature, 'test':'Chi-square', 'p_value':pval, 'effect_size_cramers_v':cramers_v(table), 'expected_cells_below_5':int((expected<5).sum())})
results_tests = pd.DataFrame(test_results)
if not results_tests.empty:
    display(results_tests.sort_values('p_value'))
if 'age' in df:
    a = df.loc[y.eq(0), 'age'].dropna()
    b = df.loc[y.eq(1), 'age'].dropna()
    if len(a) and len(b):
        stat, pval = mannwhitneyu(a,b,alternative='two-sided')
        print(f'Age Mann–Whitney U={stat:.1f}, p={pval:.4g}; group medians: {a.median():.1f} vs {b.median():.1f}')

## 6. Stratified split and leakage-safe preprocessing
**Enhancement 6:** Imputation, scaling, and one-hot encoding are fitted inside cross-validation pipelines, preventing training/test information leakage. Stratification preserves class proportions.

In [ ]:
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
], remainder='drop')
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=TEST_SIZE,random_state=SEED,stratify=y)
print('Training:', X_train.shape, 'Test:', X_test.shape)
print('Training recurrence rate:', round(y_train.mean(),3), 'Test recurrence rate:', round(y_test.mean(),3))

## 7. Train and compare classifiers
**Enhancement 7:** Adds a majority-class baseline, stratified 5-fold cross-validation, and multiple complementary evaluation metrics (F1, precision, recall, ROC AUC, PR AUC). The untouched test set is evaluated once after model specification.

In [ ]:
models = {
    'Majority baseline': Pipeline([('prep',preprocessor),('model',DummyClassifier(strategy='most_frequent'))]),
    'Logistic regression': Pipeline([('prep',preprocessor),('model',LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))]),
    'Random forest': Pipeline([('prep',preprocessor),('model',RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced',random_state=SEED,n_jobs=-1))])
}
min_class = int(y_train.value_counts().min())
if min_class < 2:
    raise ValueError('Insufficient examples of the minority class for stratified cross-validation.')
cv = StratifiedKFold(n_splits=min(5,min_class), shuffle=True, random_state=SEED)
scoring = {'f1':'f1', 'recall':'recall', 'precision':'precision', 'roc_auc':'roc_auc', 'average_precision':'average_precision'}
cv_rows=[]
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    row={'model':name}
    for metric in scoring:
        row[f'cv_{metric}_mean']=np.mean(scores[f'test_{metric}'])
        row[f'cv_{metric}_sd']=np.std(scores[f'test_{metric}'])
    cv_rows.append(row)
cv_results=pd.DataFrame(cv_rows).sort_values('cv_f1_mean',ascending=False)
display(cv_results.round(3))

In [ ]:
test_rows=[]
for name, pipeline in models.items():
    pipeline.fit(X_train,y_train)
    pred=pipeline.predict(X_test)
    proba=pipeline.predict_proba(X_test)[:,1]
    test_rows.append({'model':name, 'accuracy':accuracy_score(y_test,pred), 'precision':precision_score(y_test,pred,zero_division=0), 'recall':recall_score(y_test,pred,zero_division=0), 'f1':f1_score(y_test,pred,zero_division=0), 'roc_auc':roc_auc_score(y_test,proba), 'pr_auc':average_precision_score(y_test,proba)})
    print('\n',name,'\n',classification_report(y_test,pred,zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['No','Yes'],cmap='Blues')
    plt.title(f'{name}: confusion matrix'); plt.tight_layout(); plt.show()
test_results=pd.DataFrame(test_rows).sort_values('f1',ascending=False)
display(test_results.round(3))
fig, axes=plt.subplots(1,2,figsize=(12,5))
for name,pipeline in models.items():
    RocCurveDisplay.from_estimator(pipeline,X_test,y_test,ax=axes[0],name=name)
    PrecisionRecallDisplay.from_estimator(pipeline,X_test,y_test,ax=axes[1],name=name)
axes[0].set_title('Test ROC curves'); axes[1].set_title('Test precision–recall curves')
plt.tight_layout();plt.show()

## 8. Interpret predictors
**Enhancement 8:** Held-out permutation importance on original columns, avoiding misleading interpretation of individual one-hot encoded levels. Importance measures model reliance, not causal influence. Small test samples produce uncertain rankings.

In [ ]:
for name in ['Logistic regression','Random forest']:
    fitted=models[name]
    imp=permutation_importance(fitted,X_test,y_test,n_repeats=15,random_state=SEED,scoring='f1',n_jobs=-1)
    importance=pd.DataFrame({'feature':X.columns,'mean_f1_decrease':imp.importances_mean,'sd':imp.importances_std}).sort_values('mean_f1_decrease',ascending=False)
    print(f'Permutation importance: {name}')
    display(importance.head(12).round(3))
    top=importance.head(12).sort_values('mean_f1_decrease')
    plt.figure(figsize=(8,5));plt.barh(top['feature'],top['mean_f1_decrease'])
    plt.xlabel('Decrease in test F1 after shuffling');plt.title(name+' feature importance')
    plt.tight_layout();plt.show()

## 9. Automated findings and limitations
**Enhancement 9:** Generate a concise evidence-based summary from actual computed metrics rather than hard-coding performance claims. Use this output when writing your final interpretation.

In [ ]:
best_cv=cv_results.iloc[0]
print(f"Best cross-validated F1: {best_cv['model']} ({best_cv['cv_f1_mean']:.3f} ± {best_cv['cv_f1_sd']:.3f}).")
print(f"Observed recurrence rate: {y.mean():.1%} across {len(df)} records.")
print('Held-out comparison:')
print(test_results[['model','f1','recall','precision','roc_auc','pr_auc']].round(3).to_string(index=False))
print('\nInterpretation: Favor cross-validated performance for model selection; treat test performance as a final estimate, not a tuning target.')
print('Limitations: observational single-source data; possible selection bias; uncertain timing of staging/risk variables; small sample; no external validation; association is not causation.')
print('Clinical use would require temporal verification, external validation, calibration, fairness assessment, and prospective review.')

## 10. Enhancement log (assignment requirement)
| # | What was modified | Why | How it improves the workflow |
|---|---|---|---|
| 1 | Centralized random seed and split configuration | Ensure consistency | Reproducible reruns |
| 2 | Robust CSV/archive loader and schema/label validation | Catch malformed inputs | Clear failures and portable execution |
| 3 | Removed post-treatment response and created age groups | Limit leakage; aid EDA | More defensible diagnosis-time analysis |
| 4 | Automated stratified recurrence-rate charts | Compare clinical subgroups | Faster exploratory review |
| 5 | Added Chi-square, Cramér's V and Mann–Whitney tests | Quantify exploratory associations | Inferential evidence beyond plots |
| 6 | Preprocessing inside sklearn pipelines | Prevent preprocessing leakage | Valid cross-validation and test estimates |
| 7 | Baseline, stratified CV, ROC AUC and PR AUC | Compare models more rigorously | Class-imbalance-aware evaluation |
| 8 | Original-column permutation importance | Interpret predictions | Comparable feature reliance across models |
| 9 | Dynamically generated results summary | Avoid invented performance values | Transparent, repeatable reporting |

**Submission note:** Download the original dataset from UCI into `data/Thyroid_Diff.csv` before publishing the repository, then run all notebook cells in Colab and save the executed notebook. This notebook has not been executed against the actual dataset in this environment, so results are intentionally not fabricated.